In this notebook, I will find the shifting factor to shift the central atom to the middle of the box.
By doing so, the neighboring atoms within the cutoff distance will be guaranteed to be inside the simulation box.

In [1]:
import json
from tqdm import tqdm
import numpy as np

np.random.seed(1)

In [2]:
# Triclinic box --- hard coded from the configuration files
# Parameters of the restricted triclinic box
xlo, xhi, xy = 9.4529510586485480e2, 1.7371266170496060e3, 6.2188278662727674e0
ylo, yhi, xz = -4.0063532727924689e2, 4.1956442404085044e2, -2.4918818131154445e0
zlo, zhi, yz = -3.1011905422045612e2, 4.9472260784626945e2, 3.0074771675674690e0

# Parameters of the generalized triclinic boc
O = np.array([xlo, ylo, zlo])
A = np.array([xhi - xlo, 0, 0])
B = np.array([xy, yhi - ylo, 0])
C = np.array([xz, yz, zhi - zlo])
# Export the cell vector
with open("cell_vectors.json", "w") as f:
    json.dump(
        {"origin": O.tolist(), "v1": A.tolist(), "v2": B.tolist(), "v3": C.tolist()},
        f,
        indent=4,
    )

Let $\mathbf{x}$ be a vector in the Cartesian coordinate and $\mathbf{v}$ be a vector in triclinic box coordinate.
We can transform the two vectors through the linear transformation
\begin{equation}
    \mathbf{x} - \mathbf{x}_0 =
        \begin{bmatrix}
            x_{hi}-x_{lo} & xy & xz \\
            0 & y_{hi}-y_{lo} & yz \\
            0 & 0 & z_{hi}-z_{lo}
        \end{bmatrix}
        \mathbf{v},
\end{equation}
where $\mathbf{x}_0$ is the coordinate of the lower corner of the triclinic box.

In [3]:
# Transformation matrix
T = np.array([A, B, C]).T
# This is the center of the box in the real space
box_center = T @ (0.5 * np.ones(3)) + O

The shifting info will be stored as an array, where each row gives the displacement vector that should be applied to shift the central atom to the middle of the box.
For example, row $i$ gives the displacement vector for central atom index $i$.

In [4]:
# Now, let's shift the central atoms in the candidate configuration
# Load the coordinates of the central atoms in the real space
central_atoms = np.load("candidates_central_atom_positions.npy")
nconfigs = len(central_atoms)

# Shift
shift = np.empty((nconfigs, 3))
for ii, pos in tqdm(enumerate(central_atoms), total=nconfigs):
    shift[ii] = box_center - pos

100%|██████████████████████████████████████████████████████████| 2000/2000 [00:00<00:00, 585592.18it/s]
